# **Basics Statistics for Machine Learning**

## 1. Expectation and Variance

$$E[X]=\sum_x x\,P(X=x)\ \text{(discrete)},\quad E[X]=\int x f(x)\,dx\ \text{(continuous)}$$
$$\text{Var}(X)=E[(X-E[X])^2]=E[X^2]-(E[X])^2$$

### Worked example

Die roll: $E[X]=\sum_{k=1}^6 k\cdot\tfrac16 = \tfrac16(1+2+\dots+6)=\tfrac{21}{6}=3.5$
$$E[X^2]=\tfrac16(1+4+9+16+25+36)=\tfrac{91}{6}\approx15.17$$
$$\text{Var}(X)=15.17-3.5^2=15.17-12.25=2.92$$


## 2. Maximum Likelihood Estimation (MLE)

Given i.i.d. data $\{x_i\}$, the likelihood is $L(\theta)=\prod_i f(x_i;\theta)$.
MLE finds $\hat\theta=\arg\max_\theta L(\theta)$, usually via the
log-likelihood $\ell(\theta)=\sum_i \log f(x_i;\theta)$ since $\log$ is
monotonic and turns products into sums (numerically stable, differentiable
term-by-term). This is the theoretical justification for *why* Logistic
Regression's cross-entropy loss and Naive Bayes' probability estimates are
the "correct" objective, not just a convenient one.

`see last section of this notebook for derivations`

## 3. Central Limit Theorem (why cross-validation error estimates are trustworthy)

If $X_1,\dots,X_n$ are i.i.d. with mean $\mu$, variance $\sigma^2$, then as
$n\to\infty$:
$$\frac{\bar X_n - \mu}{\sigma/\sqrt n} \xrightarrow{d} \mathcal N(0,1)$$
This is why averaging k-fold CV error estimates gives a low-variance
estimate of true generalization error, and why confidence intervals on
model accuracy shrink as $\sqrt n$.

In [1]:
import numpy as np
from scipy import stats

# Bayes' theorem check
p_d, p_pos_given_d, p_pos_given_not_d = 0.01, 0.95, 0.05
p_pos = p_pos_given_d*p_d + p_pos_given_not_d*(1-p_d)
p_d_given_pos = (p_pos_given_d*p_d)/p_pos
print(round(p_d_given_pos, 4))   # 0.1610

# Expectation/variance of a die
x = np.arange(1,7)
p = np.full(6, 1/6)
print("E[X] =", (x*p).sum())          # 3.5
print("Var(X) =", ((x**2)*p).sum() - (x*p).sum()**2)   # 2.9167

0.161
E[X] = 3.5
Var(X) = 2.916666666666666


---

# MLE & MAP estimation
Prerequisites: Probability primer. Next: Regularization (MAP with Gaussian
prior = Ridge; MAP with Laplace prior = Lasso — derived at the end here).

## 1. Setup

Given i.i.d. data $D=\{x_1,\dots,x_n\}$ from a distribution with unknown
parameter $\theta$, the **likelihood** is $L(\theta) = \prod_{i=1}^n p(x_i\mid\theta)$.

## 2. MLE Derivation - general form


$$\hat\theta_{MLE} = \arg\max_\theta \log L(\theta) = \arg\max_\theta \sum_{i=1}^n \log p(x_i\mid\theta)$$
Take derivative w.r.t. $\theta$, set to 0, solve.


### Worked example: MLE of a Bernoulli (coin flip) parameter

Data: 7 heads, 3 tails out of $n=10$ flips, $p(x_i=1\mid\theta)=\theta^{x_i}(1-\theta)^{1-x_i}$.
$$\ell(\theta) = \sum_i \big[x_i\log\theta + (1-x_i)\log(1-\theta)\big] = 7\log\theta + 3\log(1-\theta)$$
$$\frac{d\ell}{d\theta} = \frac{7}{\theta} - \frac{3}{1-\theta} = 0 \Rightarrow 7(1-\theta)=3\theta \Rightarrow 7 = 10\theta \Rightarrow \hat\theta_{MLE}=0.7$$
This confirms the intuitive estimator (sample proportion) is exactly what
falls out of maximizing likelihood — not a coincidence, but a formal
result.

### Worked example: MLE of Gaussian mean and variance (used for residual assumption in Linear Regression)

$$\ell(\mu,\sigma^2) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i(x_i-\mu)^2$$
$$\frac{\partial \ell}{\partial \mu} = \frac{1}{\sigma^2}\sum_i(x_i-\mu)=0 \Rightarrow \hat\mu_{MLE}=\bar x$$
$$\frac{\partial \ell}{\partial \sigma^2} = -\frac{n}{2\sigma^2}+\frac{1}{2\sigma^4}\sum_i(x_i-\mu)^2=0 \Rightarrow \hat\sigma^2_{MLE}=\frac1n\sum_i(x_i-\bar x)^2$$
Numeric check with $x=\{2,4,4,4,5,5,7,9\}$: $\bar x = 40/8=5.0$,
$\hat\sigma^2 = \frac18[(2-5)^2+(4-5)^2\cdot3+(5-5)^2\cdot2+(7-5)^2+(9-5)^2]=\frac18[9+3+0+4+16]=\frac{32}{8}=4.0$.

## 3. Why Linear Regression's Least-Squares IS Maximum Likelihood

Assume $y_i = \mathbf w^T\mathbf x_i + \epsilon_i$, $\epsilon_i\sim\mathcal N(0,\sigma^2)$ i.i.d.
Then $p(y_i\mid \mathbf x_i,\mathbf w) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(y_i-\mathbf w^T\mathbf x_i)^2}{2\sigma^2}\right)$.
$$\log L(\mathbf w) = -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_i (y_i - \mathbf w^T\mathbf x_i)^2$$
Maximizing $\log L$ over $\mathbf w$ is **exactly equivalent** to minimizing
$\sum_i(y_i-\mathbf w^T\mathbf x_i)^2$ — the ordinary least-squares cost
function — since the other terms don't depend on $\mathbf w$. This is the
formal justification of why MSE is "the" loss for linear regression under
Gaussian-noise assumptions, not an arbitrary choice.

## 4. MAP Estimation — adding a prior

Bayesian view: treat $\theta$ itself as random, with prior $p(\theta)$.
$$\hat\theta_{MAP} = \arg\max_\theta p(\theta\mid D) = \arg\max_\theta \frac{p(D\mid\theta)p(\theta)}{p(D)} = \arg\max_\theta \big[\log p(D\mid\theta) + \log p(\theta)\big]$$
(dropping $p(D)$ since it doesn't depend on $\theta$).

### Derivation: Gaussian prior on weights ⇒ Ridge Regression

Let $\mathbf w \sim \mathcal N(0, \tau^2 I)$, i.e.
$\log p(\mathbf w) = -\frac{1}{2\tau^2}\|\mathbf w\|_2^2 + \text{const}$.
Combining with the Gaussian-likelihood log from §3:
$$\log p(\mathbf w\mid D) = -\frac{1}{2\sigma^2}\sum_i(y_i-\mathbf w^T\mathbf x_i)^2 - \frac{1}{2\tau^2}\|\mathbf w\|_2^2 + \text{const}$$
Maximizing this (equivalently, minimizing its negative) gives exactly:
$$\min_{\mathbf w} \sum_i(y_i-\mathbf w^T\mathbf x_i)^2 + \lambda\|\mathbf w\|_2^2,\qquad \lambda = \sigma^2/\tau^2$$
**This is precisely the Ridge Regression objective** — Ridge is MAP
estimation with a Gaussian prior on the weights. (Laplace prior gives the
$\ell_1$ penalty, i.e. Lasso — full derivation in
`03_Supervised_Learning\Regression\05_lasso_regression.ipynb`.)

In [1]:
## verifying MLE = sample mean/variance and OLS numerically

import numpy as np

x = np.array([2,4,4,4,5,5,7,9], dtype=float)
mle_mu = x.mean()
mle_var = ((x - mle_mu)**2).mean()
print(mle_mu, mle_var)   # 5.0 4.0 -- matches hand calculation

# Bernoulli MLE
heads, n = 7, 10
theta_hat = heads / n
print(theta_hat)

5.0 4.0
0.7
